# Mini Project 1: AI Text Summarizer

This notebook summarizes supplied text with Google Gemini. It lets the user enter text and choose a maximum summary length, rejects empty and overly long input, and includes five manually checked examples.

## Before you start
From the Week 1 repository folder, install the notebook dependencies with `python -m pip install -r "Mini Project 1 - AI Text Summarizer/requirements.txt"`, then open this notebook in Jupyter. Keep the API key in the repository-root `.env` file. The notebook searches its current folder and parent folders for `.env`; if you are running it elsewhere, set `GEMINI_API_KEY` in your environment instead. Never commit the key.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google import genai
from google.genai import types
import ipywidgets as widgets
from IPython.display import clear_output, display

PROJECT_ROOT = None
for folder in (Path.cwd(), *Path.cwd().parents):
    env_file = folder / ".env"
    if env_file.is_file():
        load_dotenv(env_file)
        PROJECT_ROOT = folder
        break

API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite").strip()
MAX_INPUT_CHARS = 10_000
print(f"Model: {MODEL}")
print(f"Project .env found: {PROJECT_ROOT is not None}")


In [ ]:
SYSTEM_PROMPT = (
    "You are a careful text summarizer. Use only the supplied source. "
    "Preserve important names, dates, numbers, and uncertainty. "
    "Do not add claims or infer causes that are not in the source."
)

def validate_text(text, max_chars=MAX_INPUT_CHARS):
    """Return a user-friendly error for empty or too-long text."""
    if not text or not text.strip():
        return "Please enter some text to summarize."
    if len(text) > max_chars:
        return f"The input is {len(text):,} characters; the limit is {max_chars:,}."
    return None

assert validate_text("") is not None
assert validate_text("   ") is not None
assert validate_text("Some source text") is None
assert validate_text("x" * MAX_INPUT_CHARS) is None
assert validate_text("x" * (MAX_INPUT_CHARS + 1)) is not None
print("Empty-input and length-limit checks passed.")

def summarize_text(text, max_words=60):
    """Summarize text with Gemini while honoring the requested word cap."""
    problem = validate_text(text)
    if problem:
        raise ValueError(problem)
    if not 20 <= int(max_words) <= 200:
        raise ValueError("Choose a summary length from 20 to 200 words.")
    if not API_KEY:
        raise RuntimeError("GEMINI_API_KEY is missing. Add it to the private repository-root .env file or set it in the environment.")

    prompt = (
        f"Write a concise summary of no more than {int(max_words)} words. "
        "Use only the source, retain its important facts and caveats, "
        "and do not guess. Return only the summary.\n\n"
        f"Source text:\n{text.strip()}"
    )
    client = genai.Client(api_key=API_KEY)
    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.2,
                max_output_tokens=500,
            ),
        )
        return response.text or "The model returned an empty summary."
    finally:
        client.close()

print("Input validation helpers are ready.")

## Enter text and choose the summary length

Run the next cell to display the input box and length slider. The 10,000-character limit helps keep requests manageable. Empty and oversized input are checked before any API request is sent.

In [ ]:
text_box = widgets.Textarea(
    value="",
    placeholder="Paste the text you want summarized here...",
    description="Source text:",
    layout=widgets.Layout(width="100%", height="220px"),
    style={"description_width": "initial"},
)
word_limit = widgets.IntSlider(
    value=60, min=20, max=200, step=10,
    description="Maximum summary words:",
    style={"description_width": "initial"},
)
summarize_button = widgets.Button(
    description="Summarize", button_style="success", icon="align-left"
)
result_area = widgets.Output()

def on_summarize_clicked(_):
    with result_area:
        clear_output(wait=True)
        problem = validate_text(text_box.value)
        if problem:
            print(problem)
            return
        try:
            print(summarize_text(text_box.value, word_limit.value))
        except Exception as error:
            print(f"Could not create a summary: {error}")

summarize_button.on_click(on_summarize_clicked)
display(widgets.VBox([text_box, word_limit, summarize_button, result_area]))

## Five-example factual accuracy evaluation

I generated these summaries with Gemini 3.5 Flash-Lite at temperature 0.2. I checked each output against its source for invented or altered facts. “Pass” means the summary contains no factual error; a note calls out an important omitted detail when relevant. Outputs can vary between runs.

| # | Input text | Model summary | Factual check |
|---|---|---|---|
| 1 | The Oak Street library closes at 6 p.m. Friday, 14 November for electrical maintenance and reopens Saturday at 10 a.m. Outside book returns stay available. No overdue fees apply to items due during the closure. | The Oak Street library will close at 6 p.m. on Friday, November 14, for electrical maintenance and reopen at 10 a.m. on Saturday. During the closure, book returns will remain accessible via the outside slot, and no overdue fees will be charged for items due at that time. | **Pass.** Times, reason, return option, and fee exception match the source. |
| 2 | A free science workshop for students aged 13 to 16 will take place at Greenfield Community Centre on 22 November from 1 p.m. to 4 p.m. Participants will build a small weather station in groups. Registration closes on 18 November; there are 24 places. Students should bring a notebook; materials are provided. | A free science workshop for students aged 13 to 16 will be held at Greenfield Community Centre on 22 November, from 1 p.m. to 4 p.m. Registration closes on 18 November, with places limited to 24. | **Pass, with an omission.** No facts are wrong; the summary leaves out the notebook and supplied-material details. |
| 3 | The school garden ran for eight weeks. Thirty students enrolled and 26 completed it. They planted tomatoes, beans, and spinach in six raised beds. Attendance was highest in the first month. Harvest amounts were not compared with previous years. | The school garden project ran for eight weeks with 30 enrolled students and 26 program completers. Students planted tomatoes, beans, and spinach in six raised beds, and attendance was highest during the first month. The final report does not compare harvest amounts with previous years. | **Pass.** Counts, crops, beds, attendance, and limitation match the source. |
| 4 | Rainfall was 18 mm in April and 11 mm in May. Daily per-person water use averaged 142 litres in April and 136 in May. The office advised repairing leaks. The report does not say whether rainfall caused the water-use change. | The town water office recorded 18 millimetres of rain in April and 11 millimetres in May, alongside average resident water use of 142 litres per person per day in April and 136 litres in May. Additionally, the office reminded households to repair leaking taps, and the report does not state if rainfall caused the change in water use. | **Pass.** Measurements and the no-causation caveat are retained. |
| 5 | From 3 December, Northside paper/cardboard collection changes from Tuesday to Wednesday. Plastic/metal stays Friday. The change applies only to Northside. A printed calendar can be requested by calling the waste office before 25 November. | Starting 3 December, the Northside recycling truck will collect paper and cardboard on Wednesdays instead of Tuesdays, while plastic and metal collection remains on Fridays, and residents can request a printed calendar by calling the waste office before 25 November. | **Pass, with an omission.** Collection details are accurate; the summary does not explicitly say the change applies only to Northside. |

**Evaluation result:** 5/5 summaries had no factual errors. Two summaries omitted a detail that could matter to a reader. This small sample does not prove the model is always accurate; facts should still be checked against the source.

## Model and prompt explanation

The notebook uses Google Gemini 3.5 Flash-Lite through the `google-genai` Python SDK. The system prompt asks it to summarize only the supplied source, preserve key facts and uncertainty, and avoid unsupported additions. Each request also gives a maximum word count. Temperature is set to 0.2 to keep summaries relatively consistent. The model can still omit details or make mistakes, which is why each example is checked against its source.